# 150 delivery orders: PCA on the covariance matrix

> **Before you run:** choose **Copy to Drive** at the top of this page (or **File → Save a copy in Drive**). Your copy opens in a new tab. In your copy, choose **Runtime → Run all**. Your copy is yours, so Colab runs it without a warning, and it keeps your changes.

**Principal Component Analysis** · a worked example from the StatNex teaching guide

**The story.** An analyst has 150 orders, 50 from each of three delivery apps: ZipEats (budget), FeastNow (mid-range) and UrbanBite (premium restaurants). Each order has four measures: delivery time, distance, order total and tip.

**The question.** What happens to the combined scores when each measure keeps its own units?

If you run this page without a copy, Colab first warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## Before the analysis

This analysis needs `scikit-learn`. This cell installs them. If Colab asks you to restart the session, restart it and choose **Run all** again.

In [ ]:
%pip install -q "scikit-learn"

## The table

This cell saves the table `01_basic/stratified_descriptives.csv` (150 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
No.,App,Delivery time (min),Distance (km),Order total ($),Tip ($)
1.0,ZipEats,16.0,1.5,18.41,2.07
2.0,ZipEats,24.0,1.8,22.84,2.68
3.0,ZipEats,28.0,2.4,12.55,1.77
4.0,ZipEats,19.0,1.0,14.25,0.0
5.0,ZipEats,29.0,1.9,14.64,0.21
6.0,ZipEats,26.0,1.9,16.77,1.34
7.0,ZipEats,22.0,1.2,17.94,2.51
8.0,ZipEats,30.0,2.9,9.43,0.0
9.0,ZipEats,22.0,1.3,12.85,2.08
10.0,ZipEats,26.0,2.1,19.0,1.79
11.0,ZipEats,28.0,1.2,21.85,2.97
12.0,ZipEats,17.0,1.6,9.97,0.0
13.0,ZipEats,29.0,2.5,21.83,3.46
14.0,ZipEats,26.0,2.3,14.55,0.89
15.0,ZipEats,23.0,1.3,17.27,1.06
16.0,ZipEats,23.0,1.5,21.22,1.76
17.0,ZipEats,23.0,2.2,15.01,0.95
18.0,ZipEats,22.0,1.8,17.59,2.45
19.0,ZipEats,26.0,1.4,14.39,0.86
20.0,ZipEats,12.0,0.7,18.94,0.64
21.0,ZipEats,26.0,2.5,17.0,2.43
22.0,ZipEats,14.0,2.0,9.05,1.5
23.0,ZipEats,29.0,3.2,13.19,0.0
24.0,ZipEats,20.0,2.3,11.02,0.99
25.0,ZipEats,23.0,2.1,20.25,0.63
26.0,ZipEats,20.0,1.3,16.26,1.03
27.0,ZipEats,27.0,3.2,18.54,1.01
28.0,ZipEats,23.0,2.2,14.43,2.45
29.0,ZipEats,28.0,1.8,13.04,0.67
30.0,ZipEats,19.0,2.7,9.3,0.76
31.0,ZipEats,27.0,2.3,14.7,0.02
32.0,ZipEats,27.0,2.5,11.43,0.98
33.0,ZipEats,29.0,3.1,11.19,0.55
34.0,ZipEats,24.0,1.8,19.65,0.81
35.0,ZipEats,26.0,1.6,18.99,2.62
36.0,ZipEats,18.0,1.1,20.07,0.83
37.0,ZipEats,25.0,2.2,12.95,2.23
38.0,ZipEats,58.0,2.0,15.34,1.43
39.0,ZipEats,17.0,1.0,10.77,0.0
40.0,ZipEats,31.0,2.6,20.83,0.22
41.0,ZipEats,27.0,2.2,13.66,1.75
42.0,ZipEats,18.0,1.3,12.59,0.42
43.0,ZipEats,24.0,1.4,11.63,0.55
44.0,ZipEats,29.0,1.6,12.33,1.37
45.0,ZipEats,25.0,3.1,12.61,0.74
46.0,ZipEats,24.0,1.2,14.06,1.97
47.0,ZipEats,25.0,2.3,19.86,1.34
48.0,ZipEats,23.0,2.6,12.98,2.01
49.0,ZipEats,20.0,0.7,14.33,0.7
50.0,ZipEats,26.0,1.6,9.85,1.46
51.0,FeastNow,32.0,3.9,27.47,3.59
52.0,FeastNow,26.0,3.0,29.69,2.59
53.0,FeastNow,31.0,3.2,32.42,5.02
54.0,FeastNow,33.0,3.8,32.92,3.85
55.0,FeastNow,25.0,2.1,20.56,2.82
56.0,FeastNow,37.0,4.5,21.81,3.04
57.0,FeastNow,30.0,2.7,20.8,1.18
58.0,FeastNow,28.0,2.4,24.03,2.99
59.0,FeastNow,29.0,2.5,24.28,3.67
60.0,FeastNow,35.0,4.3,28.65,3.35
61.0,FeastNow,33.0,3.9,34.57,3.4
62.0,FeastNow,27.0,2.0,27.35,3.67
63.0,FeastNow,34.0,4.0,28.64,3.25
64.0,FeastNow,26.0,3.0,23.07,2.47
65.0,FeastNow,23.0,1.7,22.11,2.36
66.0,FeastNow,30.0,2.8,26.63,3.97
67.0,FeastNow,30.0,3.2,28.15,1.64
68.0,FeastNow,33.0,2.6,31.1,3.49
69.0,FeastNow,35.0,4.4,26.51,2.67
70.0,FeastNow,29.0,2.1,26.86,2.88
71.0,FeastNow,29.0,3.2,14.45,1.71
72.0,FeastNow,42.0,5.2,26.27,2.15
73.0,FeastNow,33.0,3.5,27.28,3.67
74.0,FeastNow,32.0,2.7,28.93,2.74
75.0,FeastNow,32.0,3.3,32.57,3.64
76.0,FeastNow,41.0,4.0,18.17,0.39
77.0,FeastNow,29.0,3.6,28.68,1.87
78.0,FeastNow,36.0,3.6,34.65,2.93
79.0,FeastNow,34.0,3.3,38.95,3.52
80.0,FeastNow,33.0,2.8,25.84,2.02
81.0,FeastNow,30.0,4.3,23.9,3.4
82.0,FeastNow,32.0,3.0,20.64,2.89
83.0,FeastNow,38.0,3.9,26.52,2.64
84.0,FeastNow,35.0,3.4,23.54,2.48
85.0,FeastNow,41.0,5.1,32.35,3.14
86.0,FeastNow,31.0,4.2,19.05,1.34
87.0,FeastNow,37.0,4.6,22.4,2.18
88.0,FeastNow,37.0,3.9,18.2,3.07
89.0,FeastNow,33.0,4.2,25.6,3.21
90.0,FeastNow,31.0,3.3,21.64,0.92
91.0,FeastNow,40.0,5.4,29.35,3.64
92.0,FeastNow,39.0,4.6,21.83,2.31
93.0,FeastNow,35.0,3.6,24.4,2.99
94.0,FeastNow,37.0,3.7,12.35,1.42
95.0,FeastNow,35.0,3.8,27.95,2.41
96.0,FeastNow,35.0,4.0,22.95,1.44
97.0,FeastNow,31.0,3.9,31.9,4.72
98.0,FeastNow,33.0,3.7,28.75,4.27
99.0,FeastNow,37.0,4.6,32.45,3.61
100.0,FeastNow,33.0,3.1,27.47,2.34
101.0,UrbanBite,36.0,3.4,36.32,4.2
102.0,UrbanBite,41.0,4.7,29.45,4.63
103.0,UrbanBite,38.0,5.4,34.75,4.73
104.0,UrbanBite,48.0,7.1,54.77,8.42
105.0,UrbanBite,33.0,3.3,53.88,7.15
106.0,UrbanBite,42.0,6.3,31.78,4.1
107.0,UrbanBite,42.0,5.7,53.32,6.23
108.0,UrbanBite,42.0,4.4,22.94,3.46
109.0,UrbanBite,34.0,4.0,42.1,4.72
110.0,UrbanBite,46.0,6.0,56.02,7.98
111.0,UrbanBite,38.0,4.0,32.99,3.89
112.0,UrbanBite,53.0,7.3,49.04,8.06
113.0,UrbanBite,41.0,4.7,37.75,5.02
114.0,UrbanBite,45.0,5.1,38.8,6.29
115.0,UrbanBite,35.0,5.0,27.69,3.33
116.0,UrbanBite,46.0,6.2,61.51,8.77
117.0,UrbanBite,47.0,6.6,36.88,5.75
118.0,UrbanBite,33.0,4.2,38.32,5.06
119.0,UrbanBite,44.0,5.3,50.62,8.78
120.0,UrbanBite,31.0,2.9,68.51,9.35
121.0,UrbanBite,32.0,3.6,36.39,4.71
122.0,UrbanBite,44.0,6.9,38.89,6.6
123.0,UrbanBite,42.0,5.4,41.79,6.62
124.0,UrbanBite,39.0,3.9,46.31,6.35
125.0,UrbanBite,41.0,4.4,48.27,6.46
126.0,UrbanBite,38.0,5.2,56.79,9.08
127.0,UrbanBite,46.0,6.3,49.88,5.92
128.0,UrbanBite,38.0,4.6,44.09,6.71
129.0,UrbanBite,36.0,4.6,56.16,7.64
130.0,UrbanBite,31.0,4.3,32.96,4.45
131.0,UrbanBite,36.0,5.3,45.07,6.59
132.0,UrbanBite,38.0,4.5,57.9,7.52
133.0,UrbanBite,32.0,3.2,37.21,5.18
134.0,UrbanBite,46.0,5.8,48.05,7.86
135.0,UrbanBite,45.0,5.4,48.31,7.0
136.0,UrbanBite,38.0,4.5,39.96,5.37
137.0,UrbanBite,44.0,5.1,42.82,6.5
138.0,UrbanBite,34.0,3.7,56.6,8.21
139.0,UrbanBite,32.0,3.0,42.9,5.07
140.0,UrbanBite,29.0,3.1,35.49,5.55
141.0,UrbanBite,45.0,5.6,54.84,5.94
142.0,UrbanBite,40.0,4.7,39.08,4.98
143.0,UrbanBite,37.0,4.1,44.59,7.24
144.0,UrbanBite,42.0,5.4,44.91,5.67
145.0,UrbanBite,43.0,5.2,49.36,6.07
146.0,UrbanBite,36.0,3.4,46.85,6.8
147.0,UrbanBite,39.0,4.1,51.69,7.05
148.0,UrbanBite,40.0,5.7,44.44,4.64
149.0,UrbanBite,46.0,4.7,51.58,8.62
150.0,UrbanBite,44.0,5.7,47.37,6.35

These cells save `redundant_variables.py`: a step that no library does, written out. The analysis imports it.

In [ ]:
%%writefile redundant_variables.py
"""Which variables are exact linear combinations of the ones listed before them.

A variable that is a weighted sum of others adds no information and makes the
correlation matrix singular. No library names them, so it is written out: each
column, centred and scaled to unit length, is regressed on the columns kept so far,
and a residual sum of squares of 1e-10 or less marks it redundant. A constant column
is redundant too. The later member of a redundant pair is the one reported, so the
variable listed first survives.
"""

import numpy as np


def redundant_variables(x, names, tolerance=1e-10):
    centred = x - x.mean(axis=0)
    length = np.linalg.norm(centred, axis=0)
    length[length == 0] = 1.0
    centred = centred / length
    redundant, kept = [], []
    for j, name in enumerate(names):
        residual = centred[:, j]
        if kept:
            basis = centred[:, kept]
            residual = residual - basis @ np.linalg.lstsq(basis, residual, rcond=None)[0]
        if residual @ residual <= tolerance or x[:, j].max() == x[:, j].min():
            redundant.append(name)
        else:
            kept.append(j)
    return redundant

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from redundant_variables import redundant_variables

pd.set_option("display.width", 120)

## The data

In [ ]:
# Text and blank cells count as missing, and a case missing any variable is left out.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
variables = ["Delivery time (min)", "Distance (km)", "Order total ($)", "Tip ($)"]
label_column = []
numbers = data[variables].apply(pd.to_numeric, errors="coerce")
complete = numbers.notna().all(axis=1)
x = numbers[complete].to_numpy(float)
n, p = x.shape
cases = (data.loc[complete, label_column[0]].astype(str).tolist() if label_column
         else [str(i + 1) for i in range(n)])

summary = pd.DataFrame({"n": [n, len(data) - n, len(data)]},
                       index=["Valid cases", "Incomplete cases", "Total"])
summary["share"] = summary["n"] / len(data)
print(summary)

## Descriptive statistics

In [ ]:
# Variance and standard deviation divide by n - 1.
describe = pd.DataFrame(
    {"n": n, "mean": x.mean(axis=0), "variance": x.var(axis=0, ddof=1),
     "SD": x.std(axis=0, ddof=1), "min": x.min(axis=0), "max": x.max(axis=0)},
    index=variables,
)
print()
print(describe)

## Variance-covariance matrix

In [ ]:
# The components are found from the covariance matrix (divisor n - 1), so each
# variable counts in its own units: one with a large variance dominates the first
# component. PCA below is given the raw values, which it centres itself.
analysed = np.cov(x, rowvar=False)
prepared = x
print()
print(pd.DataFrame(analysed, index=variables, columns=variables).round(4))
print("\nLinear combinations of other variables:", redundant_variables(x, variables) or "none")

## Eigenvalues

In [ ]:
# scikit-learn's PCA finds the components by a singular value decomposition of the
# centred data, which gives exactly the eigenvalues and eigenvectors of the matrix
# above, largest first. Each eigenvalue is the variance of one component
# (explained_variance_, which divides by n - 1, as the matrix does); together they add
# up to the total variance of that matrix -- on the correlation matrix, the number of
# variables, so a component with an eigenvalue above 1 carries more than one
# variable's worth. The share is each eigenvalue over that total.
pca = PCA(svd_solver="full").fit(prepared)
eigenvalues = pca.explained_variance_
vectors = pca.components_.T
share = eigenvalues / eigenvalues.sum()
eigen = pd.DataFrame({"eigenvalue": eigenvalues, "share": share, "cumulative": share.cumsum()},
                     index=range(1, p + 1))
print()
print(eigen)

## Eigenvectors and loadings

In [ ]:
# An eigenvector (one column of weights per component) and its negative describe the
# same component, so each one is flipped to make its elements sum to a positive
# number; packages pick the sign arbitrarily. A loading is the correlation between a
# variable and a component: the eigenvector times the square root of its eigenvalue.
# Read the loadings, not the eigenvectors, to name a component: the variables with
# the largest loadings, positive and negative, are what it contrasts.
signs = np.where(vectors.sum(axis=0) < 0, -1.0, 1.0)
vectors = vectors * signs
keep = min(8, p)
components = [f"PC{j + 1}" for j in range(keep)]
eigenvectors = pd.DataFrame(vectors[:, :keep], index=variables, columns=components)
loadings = eigenvectors * np.sqrt(eigenvalues[:keep])
# On the covariance matrix the loading is also divided by the variable's SD, so it
# stays a correlation, between -1 and 1.
loadings = loadings.div(x.std(axis=0, ddof=1), axis=0)
print()
print(eigenvectors.round(4))
print()
print(loadings.round(4))

## Component scores

In [ ]:
# Each case's centred values (value minus mean) times the eigenvectors: PCA's
# transform(), with the same sign flips. A case's scores place it on the map.
scores = pd.DataFrame(pca.transform(prepared)[:, :keep] * signs[:keep], index=cases,
                      columns=components)
print()
print(scores.round(4))

## What the result says

Principal component analysis (PCA) builds new scores, called components, that keep as much variation as possible. With the covariance matrix each measure keeps its own units, so a measure with a large variance (spread) counts for more.

Order total has by far the largest variance, 192, against 68.6 for delivery time, 5.72 for tip and 2.22 for distance. So the first component is mostly order total: its weight in the eigenvector, the recipe that builds the component, is 0.887. This component has a contribution rate of 0.882, which means it holds about 88% of the variation. The second component (0.114) is mostly delivery time, with a weight of 0.892.

Distance hardly counts, only because kilometres give small numbers. When measures have different units, as here, the correlation matrix is the fairer choice.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.